# Install Packages

In [ ]:
# pip install git+https://github.com/dnth/rag-datakit.git

In [ ]:
# !pip install ipywidgets
# !pip install python-dotenv


  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)


# Load ENV

In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get token from environment
token = os.getenv("HF_TOKEN")
login(token=token)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


# Load SSF Data

In [2]:
from datasets import load_dataset

dataset = load_dataset("dnth/ssf-dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['Sector', 'Track', 'Job Role', 'Job Role Description', 'Performance Expectation'],
        num_rows: 1885
    })
})

In [3]:
dataset["train"][0]

{'Sector': 'Accountancy',
 'Track': 'Assurance',
 'Job Role': 'Audit Associate / Audit Assistant Associate',
 'Job Role Description': 'The Audit Associate/Audit Assistant Associate undertakes specific stages of audit work under supervision. He/She begins to appreciate the underlying principles behind the tasks assigned to him as part of the audit plan. He is also able to make adjustments to the application of skills to improve the work tasks or solve non-complex issues. The Audit Associate/Audit Assistant Associate operates in a structured work environment. He is able to build relationships, work in a team and identify ethical issues with reference to the code of professional conduct and ethics. He is able to select and apply from a range of known solutions to familiar problems and takes responsibility for his own learning and performance. He is a trustworthy and meticulous individual.',
 'Performance Expectation': 'In accordance with: Singapore Standards on Auditing, Ethics Pronouncem

# Synthetic Data Generation Setup

In [4]:
import os
from distilabel.models import OpenAILLM, TransformersLLM

# llm = TransformersLLM(
#     model="Qwen/Qwen3-4B-Instruct-2507",
#     device_map="auto",
#     torch_dtype="float16",
# )

llm = OpenAILLM(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [ ]:
context_easy = """
You are an HR assistant tasked with creating job descriptions based on the Singapore SkillsFuture Framework. For each job, create **positive** and **easy negative** variations.

### Positive Descriptions
- Capture the **core responsibilities**, **skills**, and **expectations** of the role.
- Rephrase sentences naturally while keeping the same meaning.
- Start with: "The <job role>."
- Ensure the description is **clear, professional**, and **readable**.
- Examples:
   - *Positive Example*: "The Operations Manager oversees daily operations and ensures the company's goals are met efficiently."
   - *Rephrased Example*: "The Operations Manager is responsible for managing daily activities and ensuring that the company's objectives are efficiently met."

### Easy Negative Descriptions
- Generate a description that is **clearly unrelated** to the original, with no overlap in skills, sector, or responsibilities.
- The job role should be **completely different** in terms of function, industry, or job focus.
- Example: 
   - *Original Role*: "The Operations Manager ensures smooth day-to-day operations."
   - *Negative Example*: "The Graphic Designer creates digital content for marketing campaigns."

### General Instructions
- Ensure the descriptions are **clear**, **professional**, and **complete**.
- Ensure descriptions are **realistic**, **readable**, and **varied**.
- Avoid **generic** or **incorrect** descriptions.
- The output will be used for fine-tuning in Distilabel for sentence-pair generation.
"""

context_hard = """
You are an HR assistant tasked with creating job descriptions based on the Singapore SkillsFuture Framework. For each job, create **positive** and **hard negative** variations.

### Positive Descriptions
- Capture the **core responsibilities**, **skills**, and **expectations** of the role.
- Rephrase sentences naturally while keeping the same meaning.
- Start with: "The <job role>."
- Ensure the description is **clear, professional**, and **readable**.
- Examples:
   - *Positive Example*: "The Operations Manager oversees daily operations and ensures the company's goals are met efficiently."
   - *Rephrased Example*: "The Operations Manager is responsible for managing daily activities and ensuring that the company's objectives are efficiently met."

### Hard Negative Descriptions
- Generate a description that is **similar in appearance but semantically different**.
- Strategies:
   - Use a **different seniority level** within the same sector (e.g., Senior Manager → Manager).
   - **Shift the sector** or focus (e.g., Engineering → Operations).
   - **Substitute similar skills in different sectors** (e.g., Finance in Banking → Healthcare).
   - **Similar role, different sector** (e.g., Project Manager in IT → Project Manager in Construction).
- Keep some **similar keywords** to make the negative harder to distinguish.
- Example:
   - *Original Role*: "The Senior Operations Manager oversees high-level operations and strategic implementation."
   - *Hard Negative Example*: "The Senior IT Project Manager manages technology projects, focusing on systems integration."

### General Instructions
- Ensure the descriptions are **clear**, **professional**, and **complete**.
- Ensure descriptions are **realistic**, **readable**, and **varied**.
- Avoid **generic** or **incorrect** descriptions.
- The output will be used for fine-tuning in Distilabel for sentence-pair generation.
"""


In [14]:
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import GenerateSentencePair

with Pipeline(name="generate") as pipeline:
    load_dataset = LoadDataFromHub(
        num_examples=5,  # Limit to 10 examples for demo - increase for production datasets
        use_cache=False,  # Disable caching to ensure fresh data generation each run
        output_mappings={"Job Role Description": "anchor"},  # Map original column to 'anchor' for triplet generation
    )
    generate_retrieval_pairs_easy = GenerateSentencePair(
        name="easy_triplets_paraphrase",
        triplet=True,  # Generate anchor-positive-negative triplets for embedding training
        hard_negative=False,  # Use easier negatives rather than hard negatives
        action="paraphrase",  # Focus on paraphrasing for positive examples
        llm=llm,  # Use the LLM configured above (local Qwen or OpenAI)
        input_batch_size=2,  # Process 10 examples at once for efficiency
        #context=context,  # Provide the context instructions for generation quality
        context=context_easy,  # Provide the context instructions for generation quality
    )
    generate_retrieval_pairs_hard = GenerateSentencePair(
        name="hard_triplets_paraphrase",
        triplet=True,  
        hard_negative=True,  
        action="paraphrase",  
        llm=llm,  
        input_batch_size=2,  
        #context=context,  
        context=context_hard,  
    )
    
    generate_retrieval_pairs_hard_semantic = GenerateSentencePair(
        name="hard_triplets_semantic",
        triplet=True,  # enable negative output
        hard_negative=True,  # negative sentence is crafted to be semantically close to the anchor, making it more challenging for models to distinguish between the positive and negative pairs. This is useful for fine-tuning models to improve their discrimination capabilities.
        action="semantically-similar",  
        llm=llm,  
        input_batch_size=2,  
        #context=context,  
        context=context_hard,  
    )

    load_dataset.connect(generate_retrieval_pairs_easy, generate_retrieval_pairs_hard, generate_retrieval_pairs_hard_semantic)

In [15]:
distiset = pipeline.run(
    use_cache=False,
    parameters={
        load_dataset.name: {
            "repo_id": "dnth/ssf-dataset",
            "split": "train",
        },
        "easy_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": 256}}
        },
        "hard_triplets_paraphrase": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": 256}}
        },
        "hard_triplets_semantic": {
            "llm": {"generation_kwargs": {"temperature": 0.6, "max_new_tokens": 256}}
        },
    }
)

[08/19/25 11:59:43] INFO     ['distilabel.pipeline'] 📝 Pipeline data will be written to               ]8;id=548770;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=746777;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1015\1015]8;;\
                             '/home/frank123/.cache/distilabel/pipelines/generate/4c0dec9e1fa6a01a970c             
                             cd02c94a02229d675e7d/executions/6bb66ba7bc8fac58cebe6615e29c00b6d68242ab/             
                             data/steps_outputs'                                                                   

                    INFO     ['distilabel.pipeline'] ⌛ The steps of the pipeline will be loaded in    ]8;id=914747;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=949960;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1046\1046]8;;\
                             stages:                                                                               
                              * Legend: 🚰 GeneratorStep 🌐 GlobalStep 🔄 Step                                     
                              * Stage 0:                                                                           
                                - 🚰 'load_data_from_hub_0'                                                        
                                - 🔄 'easy_triplets_paraphrase'                                                    
                                - 🔄 'hard_triplets_paraphrase'                                                    
                                - 🔄 'hard_triplets_semantic'                                                      

[08/19/25 11:59:35] INFO     ['distilabel.pipeline'] ⏳ Waiting for all the steps of stage 0 to        ]8;id=884197;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=925730;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1382\1382]8;;\
                             load...                                                                               

[08/19/25 11:59:38] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 3/4                 ]8;id=78964;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=804466;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 0/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_semantic' replicas: 1/1                                             

[08/19/25 11:59:40] INFO     ['distilabel.pipeline'] ⏳ Steps from stage 0 loaded: 4/4                 ]8;id=529178;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py\base.py]8;;\:]8;id=208342;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/base.py#1418\1418]8;;\
                              * 'load_data_from_hub_0' replicas: 1/1                                               
                              * 'easy_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_paraphrase' replicas: 1/1                                           
                              * 'hard_triplets_semantic' replicas: 1/1                                             

...

[08/19/25 11:59:54] INFO     ['distilabel.step.hard_triplets_paraphrase'] 📨 Step               ]8;id=912503;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=735594;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py#290\290]8;;\
                             'hard_triplets_paraphrase' sending batch 2 to output queue                            

                    INFO     ['distilabel.step.hard_triplets_paraphrase'] 🏁 Finished running   ]8;id=876924;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py\step_wrapper.py]8;;\:]8;id=401665;file:///home/frank123/miniforge3/envs/distilable/lib/python3.13/site-packages/distilabel/pipeline/step_wrapper.py#129\129]8;;\
                             step 'hard_triplets_paraphrase' (replica ID: 0)                                       

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [16]:
distiset

Distiset({
    easy_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 5
        })
    })
    hard_triplets_semantic: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 5
        })
    })
    hard_triplets_paraphrase: DatasetDict({
        train: Dataset({
            features: ['Sector', 'Track', 'Job Role', 'anchor', 'Performance Expectation', 'positive', 'negative', 'distilabel_metadata', 'model_name'],
            num_rows: 5
        })
    })
})

In [17]:
distiset["hard_triplets_semantic"]["train"][-1]

{'Sector': 'Accountancy',
 'Track': 'Business Valuation',
 'Job Role': 'Business Valuation Associate / Business Valuation Executive',
 'anchor': 'The Business Valuation Associate/Business Valuation Executive has significant responsibility for execution of deliverables. He/She needs to work hands-on on the valuation analysis. He is expected to adhere to standards of ethics and maintain quality assurance in processes. The Business Valuation Associate/Business Valuation Executive participates in business development and stakeholder interaction. He has minimal experience and is expected to embark on a steep learning curve to acquire various skills and expertise in business valuation including valuation of intangible assets. He possesses strong time management and communication skills.',
 'Performance Expectation': "In accordance with the International Valuation Standards Council's Code of Ethical Principles for Professional Valuers",
 'positive': 'The Business Valuation Associate/Business 

In [18]:
hard_triplets_semantic_df = distiset["hard_triplets_semantic"]["train"].to_pandas()
hard_triplets_semantic_df

,Sector,Track,Job Role,anchor,Performance Expectation,positive,negative,distilabel_metadata,model_name
0,Accountancy,Assurance,Audit Associate / Audit Assistant Associate,The Audit Associate/Audit Assistant Associate ...,In accordance with: Singapore Standards on Aud...,The Audit Associate/Audit Assistant Associate ...,The Audit Manager oversees various stages of f...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
1,Accountancy,Assurance,Audit Manager,The Audit Senior Manager/Audit Manager manages...,In accordance with: Singapore Standards on Aud...,The Audit Senior Manager/Audit Manager is resp...,The Audit Associate Manager oversees a collect...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
2,Accountancy,Assurance,Audit Partner / Audit Director,The Audit Partner/Audit Director is a transfor...,In accordance with: Singapore Standards on Aud...,The Audit Partner/Audit Director is a visionar...,The Audit Manager is a tactical supervisor who...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
3,Accountancy,Assurance,Audit Senior,The Audit Senior is expected to team lead vari...,In accordance with: Singapore Standards on Aud...,The Audit Senior is responsible for leading a ...,The Audit Manager is tasked with overseeing mu...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini
4,Accountancy,Business Valuation,Business Valuation Associate / Business Valuat...,The Business Valuation Associate/Business Valu...,In accordance with the International Valuation...,The Business Valuation Associate/Business Valu...,The Business Valuation Manager/Business Valuat...,{'raw_input_hard_triplets_semantic': [{'conten...,gpt-4o-mini


In [19]:
distiset.push_to_hub("frankwong2001/ssf-dataset-synthetic")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md: 0.00B [00:00, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md: 0.00B [00:00, ?B/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]